# libraries

In [2]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_excel('monthly sales data.xlsx')

In [ ]:
data.head(5)
data.tail()
data.dtypes
data.isnull()
import matplotlib.pyplot as plt
%matplotlib inline
data.set_index('Month',inplace=True)

In [ ]:
, index_col='date', parse_dates=True
from statsmodels.tsa.stattools import adfuller
test_result=adfuller(data['Sales'])


In [ ]:
def adfuller_tests(sales):
    result=adfuller(sales)
    labels = ['ADF Test Stats','p-value','#lags used','Nr of obvservations used']
    for value,label in zip(result,labels):
        print(label+' : '+str(value) )
    if result[1] <= 0.05:
        print("strong evidence against null hypotheses,reject null hypo,data has no root unit,it is stationary")
    else:
        print("weak evidence against null hypothese, time series has unit root, depicting it is not stationary")

In [ ]:
 adfuller_tests(data['Sales'])
# ADF TEST STATS: tells you how strongly the data rejects or supports the null hypothesis
#                 The more negative this number, the more evidence you have that the data is stationary.
# P value: indicates the probability of getting the test results if the null hypothesis (non-stationarity) is true.
#           p-value (less than 0.05) means data is STATIONARY.
# LAGS : show how many past values impact the current step. 0 use means it didnt consider any prev value to adjust autocorel.

In [3]:
data['Sales first difference'] = data['Sales'] - data['Sales'].shift(1)
data['Sales'].shift(1)
data['Seasonal first difference'] = data['Sales'] - data['Sales'].shift(12)


NameError: name 'data' is not defined

In [ ]:
adfuller_tests(data['Seasonal first difference'].dropna())
data['Seasonal first difference'].plot()


In [4]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [ ]:
fig = plt.figure(figsize=(12,8))
ax1 = fig.add_subplot(211)
fig = sm.graphics.tsa.plot_acf(data['Seasonal first difference'].dropna(), lags=10, ax=ax1)
ax2 = fig.add_subplot(212)
fig = sm.graphics.tsa.plot_pacf(data['Seasonal first difference'].dropna(), lags=10, ax=ax2)

plt.show()
# ACF: ACF plot helps identify how many lags are significant for the moving average (MA) component
# PACF:PACF plot shows the direct relationship between a lag and the current value, after removing the influence of intermediate lags.
    # AR (P) = 1, The PACF plot shows a significant spike at lag 1, which suggests the data has an autocorrelation with only the first lag
    #  D = 1, One level of differencing was enough to make the data stationary.
    # MA (Q) = 1, The ACF plot shows a significant spike at lag 1, and after that, the autocorrelations mostly stay within the confidence interval

In [ ]:
model=sm.tsa.statespace.SARIMAX(data['Sales'],seasonal_order=(1,1,1,12))
results=model.fit()
# SARIMAX is specifically designed to handle seasonal patterns in your data, where ARIMA FAILS!
# (P,D,Q,s) = (1,1,1,12) given by above values and AUTO_SARIMAX. WHICH GIVES THE BEST MAE & MAPE

In [ ]:
data['forecast']=results.predict(start=25,end=36,dynamic=True)
data[['Sales','forecast']].plot(figsize=(12,8))
# Predict the forecast accuracy by mapping the forecast to orginal data to check accuracy.

In [ ]:
from pandas.tseries.offsets import DateOffset
future_dates=[data.index[-1]+ DateOffset(months=x)for x in range(0,15)]
# list of future dates extending the time series by 14 months

In [ ]:
future_datest_data=pd.DataFrame(index=future_dates[1:],columns=data.columns)
# empty DataFrame to hold data for these future dates,skipping the first one which is the last date of the original data.

In [ ]:
future_data=pd.concat([data,future_datest_data])
# Concat current data and future data.

In [ ]:
future_data['forecast']= results.predict(start = 36, end = 65, dynamic=True)
future_data[['Sales', 'forecast']].plot(figsize=(12,8))